In [3]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

# assistant = RAGBase(
#     index=sqlite_index,
#     llm_client=openai_client,
# )

# answer = assistant.rag("Can I still join the course after it started?")
# print(answer)

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
)

response.output_text

'Possibly—if enrollment is still open, you can usually join. Check the course page for registration details or contact the instructor/organizer to ask whether late enrollment is allowed. If you tell me which course you mean, I can help you figure out the next step.'

In [4]:
from sqlitesearch import TextSearchIndex


index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [5]:
def search(query):
    boost_dict = {
        "question": 3.0,
        "section": 0.5
    }

    filter_dict = {
        "course": "llm-zoomcamp"
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": (
        "Search the FAQ database for entries "
        "matching the given query."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Search query text to look up "
                    "in the course FAQ."
                )
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    }
}

In [7]:
response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment joining course newly discovered course"}', call_id='call_1y4Pmhi97DeTYGB3yxKSN6Ac', name='search', type='function_call', id='fc_0fee07cbeb034945006adef7d5861c87d0b98afc703f92b650', async_=None, caller=None, namespace=None, status='completed')]

In [8]:
call = response.output[0]

In [9]:
import json


args = json.loads(call.arguments)

In [10]:
results = search(**args)

In [11]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through th

In [12]:
result_json = json.dumps(results, indent=2)

In [13]:
messages.extend(response.output)

In [14]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
})

In [15]:
response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes—you can still join and start learning. If you want a certificate, you’ll need to submit your capstone project while submissions are open for a live cohort.'

In [16]:
instructions = """
You're a course teaching assistent.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore
""".strip()

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [18]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-6-luna",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it += 1

    if has_function_calls == False:
        break


iteration #1...
ASSISTANT:
Yes—you can still join and work through the course materials. If you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while submissions are open for a live cohort.


In [19]:
def agent_loop(instructions, question, model="gpt-6-luna") -> str:
    messages = [
        {
            "role": "developer",
            "content": instructions
        },
        {
            "role": "user",
            "content": question
        }
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")

        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                call_output = make_call(item)

                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                last_answer = item.content[0].text
                print(last_answer)

        it += 1

        if not has_function_calls:
            break

    return last_answer


In [20]:
agent_loop(
    instructions,
    "How do I run Olama locally?"
)


iteration #1...
iteration #2...
iteration #3...
If you mean **Ollama**, you can run a model locally like this:

1. **Install Ollama** from [ollama.com/download](https://ollama.com/download). On Linux, you can also run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Download and start a model** in a terminal:
   ```bash
   ollama run llama3
   ```
   The first run downloads the model, then opens an interactive chat. You can replace `llama3` with another model name supported by Ollama.

3. **Check that the local server is responding**:
   ```bash
   curl http://localhost:11434/api/tags
   ```
   Ollama’s local API normally listens on port `11434`.

To call the model from Python, install the client and use it like this:

```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model="llama3",
    messages=[{"role": "user", "content": "Hello!"}],
)
print(response["message"]["content"])
```

If you get “connection refused,” make sure Oll

'If you mean **Ollama**, you can run a model locally like this:\n\n1. **Install Ollama** from [ollama.com/download](https://ollama.com/download). On Linux, you can also run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. **Download and start a model** in a terminal:\n   ```bash\n   ollama run llama3\n   ```\n   The first run downloads the model, then opens an interactive chat. You can replace `llama3` with another model name supported by Ollama.\n\n3. **Check that the local server is responding**:\n   ```bash\n   curl http://localhost:11434/api/tags\n   ```\n   Ollama’s local API normally listens on port `11434`.\n\nTo call the model from Python, install the client and use it like this:\n\n```bash\npip install ollama\n```\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model="llama3",\n    messages=[{"role": "user", "content": "Hello!"}],\n)\nprint(response["message"]["content"])\n```\n\nIf you get “connection refused,” make sure Ollama is runni

In [21]:
agent_loop(
    instructions,
    "I just discovered the course. Can I still join it?"
)

iteration #1...
iteration #2...
iteration #3...
Yes—you can still join and start learning. You don’t need to register to access the course or submit homework while its form is open.

If you want a certificate, though, you’ll need to complete the capstone project and required peer reviews while a live cohort is accepting submissions. Would you like help finding the course materials or checking the current submission deadlines?


'Yes—you can still join and start learning. You don’t need to register to access the course or submit homework while its form is open.\n\nIf you want a certificate, though, you’ll need to complete the capstone project and required peer reviews while a live cohort is accepting submissions. Would you like help finding the course materials or checking the current submission deadlines?'

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
iteration #2...
iteration #3...
I couldn’t find information about “Queen’s Gambit” in the course FAQ, so I can’t answer that here. Is there a course-related area you’d like to explore?


'I couldn’t find information about “Queen’s Gambit” in the course FAQ, so I can’t answer that here. Is there a course-related area you’d like to explore?'

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
iteration #2...
iteration #3...
Yes—you can join and start learning. You can work through the materials at your own pace, and you don’t need to register to begin.

If you want a certificate, though, you’ll need to complete the capstone project and required peer reviews while a live cohort is still accepting submissions. Would you like help finding the course materials or figuring out the certificate requirements?


'Yes—you can join and start learning. You can work through the materials at your own pace, and you don’t need to register to begin.\n\nIf you want a certificate, though, you’ll need to complete the capstone project and required peer reviews while a live cohort is still accepting submissions. Would you like help finding the course materials or figuring out the certificate requirements?'

In [24]:
def search(query):
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [25]:
from toyaikit.tools import Tools


agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [26]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries
    matching the given query.
    """

    return index.search(
        query,
        num_results=5,
        boost_dict={
            "question": 3.0,
            "section": 0.5
        },
        filter_dict={
            "course": "llm-zoomcamp"
        }
    )

In [27]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [28]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries\nmatching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [32]:
from toyaikit.chat import IPythonChatInterface, OpenAIClient
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback
)

chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-6-luna"
    )
)

In [33]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [34]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results\nand then perform more searches.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"How do I run Olama locally? Ollama local install run model"}', call_id='call_3iboFDWOOWSD78mZiHDkh7fT', name='search', type='function_call', id='fc_01448d6fd71efc28006adef8f3516c87d0b45faa5910d540bf', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_3iboFDWOOWSD78mZiHDkh7fT',
  '

In [35]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback
)

-> Response received


-> Response received


In [36]:
result.cost

CostInfo(input_cost=Decimal('0.0004389'), output_cost=Decimal('0.0002225'), total_cost=Decimal('0.0006614'))

In [37]:
runner.run()

-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user